# AIML-02 on Google Colab
Runtime -> Change runtime type -> **T4 GPU** (strongly recommended). Upload `AIML-02.zip` in the Files panel first.

In [ ]:
!unzip -o -q AIML-02.zip -d . && pip install -q medmnist monai nibabel pydicom pyyaml dvc fastapi uvicorn

## 1. Run the whole pipeline with DVC (prepare data -> train classifier -> train 3D U-Net -> Grad-CAM + demo DICOMs)
Takes roughly 15-30 min on a T4. If `dvc repro` gives trouble, run the fallback cell instead.

In [ ]:
%cd /content/AIML-02
!git init -q && git config user.email colab@example.com && git config user.name colab && dvc init -q
!dvc repro

In [ ]:
# FALLBACK (only if the DVC cell above failed): same steps without DVC
# !python run_all.py

## 2. Results

In [ ]:
!cat results/classifier_metrics.json | head -40; echo; cat results/seg_metrics.json | head -20

In [ ]:
from IPython.display import Image, display
for f in ['results/classifier_auroc.png','results/gradcam_examples.png','results/seg_examples/case_1.png']:
    display(Image(f))

## 3. DICOM service demo
Option A (real DICOM server): install Orthanc, upload the demo studies, let the worker pull + analyse them.

In [ ]:
!apt-get install -y -q orthanc > /dev/null 2>&1; mkdir -p /content/orthanc-db
import subprocess, time
subprocess.Popen('Orthanc /content/AIML-02/configs/orthanc_colab.json > /content/orthanc.log 2>&1', shell=True)
time.sleep(8)
!curl -s http://localhost:8042/system | head -c 300

In [ ]:
!python scripts/upload_to_orthanc.py --dir samples/dicom --url http://localhost:8042
!python -m service.worker --orthanc http://localhost:8042 --once

In [ ]:
# Option B (if Orthanc would not start): read the same DICOM files from a folder
# !python -m service.worker --dicom-dir samples/dicom --once

## 4. Radiologist viewer (React) + audit log
Starts the API, opens the viewer inline, and records two clearly-labelled *automated demo* decisions so the audit log has entries.

In [ ]:
import subprocess, time, requests, json
subprocess.Popen(['uvicorn','service.api:app','--port','8000'], cwd='/content/AIML-02')
time.sleep(6)
studies = requests.get('http://localhost:8000/api/studies').json()
for s, dec in zip(studies[:2], ['approve','reject']):
    r = requests.post(f"http://localhost:8000/api/studies/{s['id']}/decision", json={'decision': dec, 'reviewer': 'Demo Reviewer (automated test)', 'comment': 'automated demo decision'})
    print(s['patient_id'], dec, r.status_code)
audit = requests.get('http://localhost:8000/api/audit').json()
print('audit chain valid:', audit['chain_valid'], '| entries:', len(audit['entries']))
json.dump(audit, open('results/audit_log_demo.json','w'), indent=2)
from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=900)

## 5. Download everything
Unzip the download into your local `AIML-02` folder (merge/replace).

In [ ]:
!cd /content/AIML-02 && zip -qr /content/aiml02_outputs.zip . -x '.git/*' 'data/medmnist/*' 'data/msd/*' 'data/processed/*' '.dvc/cache/*' '.dvc/tmp/*' 'runtime/*' '*__pycache__*' '*.ipynb_checkpoints*'
from google.colab import files; files.download('/content/aiml02_outputs.zip')